[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/01_eda.ipynb)

# 01 — Exploratory Data Analysis

**Purpose.** Look at what `00_simulation.ipynb` produced.

**Inputs.** The four simulation artifacts, read-only, at the paths in
`cfg.simulation.output`.

**Outputs.** *Plots and tables only.* Nothing here writes to `data/` or
`models/`.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/simulation.yaml` resolve the
same way they do for `task simulation`. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not
already ship. Note that `data/` is DVC-tracked and therefore *not* part of the
clone.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = "."  # project root inside the repository

# (import name, pip name). Colab already ships numpy, pandas, matplotlib and
# seaborn, so only these are installed.
COLAB_PACKAGES = [("hydra", "hydra-core")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = (checkout / SUBDIR).resolve()
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from omegaconf import OmegaConf

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

## 2. Load the artifacts

Read-only. If a file is missing, run `00_simulation.ipynb` or `task simulation`.

In [ ]:
out = cfg.simulation.output

ue = pd.read_csv(out.ue_file)
manifest = json.loads(Path(out.manifest_file).read_text(encoding="utf-8"))
with np.load(out.radio_map_file, allow_pickle=False) as archive:
    radio = {key: archive[key] for key in archive.files}
mdt = pd.read_csv(out.mdt_file)

bands = [str(label) for label in radio["band_label"]]
tx_names = [str(name) for name in radio["tx_name"]]
n_rows, n_cols = int(radio["n_rows"]), int(radio["n_cols"])
cell_size = float(radio["cell_size_m"])
extent = [
    float(radio["origin_x"]),
    float(radio["origin_x"]) + n_cols * cell_size,
    float(radio["origin_y"]),
    float(radio["origin_y"]) + n_rows * cell_size,
]

pd.DataFrame(
    [
        {"artifact": "ue_positions", "rows": len(ue), "cols": ue.shape[1]},
        {"artifact": "mdt", "rows": len(mdt), "cols": mdt.shape[1]},
        {"artifact": "radio_map", "rows": str(radio["rsrp_dbm"].shape), "cols": "[band, tx, row, col]"},
    ]
)

## 3. Scenario manifest

In [ ]:
pd.DataFrame(
    {"value": {**manifest["grid"], "scenario_id": manifest["scenario_id"], "seed": manifest["seed"]}}
)

In [ ]:
pd.DataFrame({"value": manifest["area"]})

In [ ]:
pd.DataFrame(manifest["density"]["hotspots"])

In [ ]:
pd.DataFrame(
    {
        "value": {
            "buildings_removed": len(manifest["perturbation"]["removed"]),
            "buildings_jittered": len(manifest["perturbation"]["jittered"]),
            "n_intervals": manifest["time"]["n_intervals"],
            "ue_rows": manifest["ue"]["rows"],
            **manifest["time"]["spec"],
        }
    }
)

## 4. UE population

In [ ]:
ue.head()

In [ ]:
ue.describe().T

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.0))
for component, part in ue.groupby("component"):
    label = "background" if component == -1 else f"hotspot {component}"
    ax.scatter(part["x"], part["y"], s=5, alpha=0.35, label=label)
ax.set_aspect("equal")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("UE positions by mixture component")
ax.legend(markerscale=3)
plt.show()

In [ ]:
per_interval = ue.groupby("t_s").size()

fig, ax = plt.subplots()
ax.plot(per_interval.index / 3600.0, per_interval.to_numpy(), lw=1.0)
ax.set_xlabel("time [h]")
ax.set_ylabel("UEs")
ax.set_title("UEs per interval")
plt.show()

In [ ]:
mass = pd.DataFrame(
    manifest["time"]["component_mass"],
    index=np.asarray(manifest["time"]["t_s"]) / 3600.0,
)
mass.columns = ["background"] + [f"hotspot {i}" for i in range(mass.shape[1] - 1)]

fig, ax = plt.subplots()
mass.plot(ax=ax, lw=1.0)
ax.set_xlabel("time [h]")
ax.set_ylabel("share of population")
ax.set_title("Mixture mass over time")
plt.show()

In [ ]:
counts = np.zeros((n_rows, n_cols))
np.add.at(counts, (ue["cell_row"].to_numpy(), ue["cell_col"].to_numpy()), 1.0)

fig, ax = plt.subplots(figsize=(8.0, 6.5))
image = ax.imshow(counts, origin="lower", extent=extent, aspect="equal")
fig.colorbar(image, ax=ax, label="UEs")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("UE count per grid cell, all intervals")
plt.show()

## 5. Transmitter layout

In [ ]:
sectors = pd.json_normalize(
    OmegaConf.to_container(cfg.simulation.transmitters.sectors, resolve=True)
)
sectors

In [ ]:
boresight_m = 90.0

fig, ax = plt.subplots(figsize=(7.5, 7.0))
ax.scatter(ue["x"], ue["y"], s=2, alpha=0.12, color="grey", label="UE")
ax.scatter(
    sectors["x"], sectors["y"], marker="^", s=110, color="crimson", zorder=3, label="sector"
)
for _, row in sectors.iterrows():
    angle = np.radians(row["azimuth_deg"])
    ax.plot(
        [row["x"], row["x"] + boresight_m * np.cos(angle)],
        [row["y"], row["y"] + boresight_m * np.sin(angle)],
        color="crimson",
        lw=1.2,
        zorder=3,
    )
ax.set_aspect("equal")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("Sector positions and boresights")
ax.legend(markerscale=2)
plt.show()

## 6. Radio map

In [ ]:
rsrp = radio["rsrp_dbm"]

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    best_server = np.stack([np.nanmax(rsrp[i], axis=0) for i in range(len(bands))])

fig, axes = plt.subplots(1, len(bands), figsize=(5.2 * len(bands), 4.6), constrained_layout=True)
for ax, index, band in zip(np.atleast_1d(axes), range(len(bands)), bands):
    image = ax.imshow(best_server[index], origin="lower", extent=extent, aspect="equal")
    fig.colorbar(image, ax=ax, label="RSRP [dBm]")
    ax.set_title(band)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
fig.suptitle("Best-server RSRP")
plt.show()

In [ ]:
fig, ax = plt.subplots()
for index, band in enumerate(bands):
    values = best_server[index]
    ax.hist(values[np.isfinite(values)], bins=60, histtype="step", lw=1.4, label=band)
ax.set_xlabel("best-server RSRP [dBm]")
ax.set_ylabel("cells")
ax.set_title("Best-server RSRP distribution")
ax.legend()
plt.show()

In [ ]:
pd.DataFrame(
    {
        "cells_reached": np.isfinite(best_server).mean(axis=(1, 2)),
        "rsrp_min": np.nanmin(best_server, axis=(1, 2)),
        "rsrp_median": np.nanmedian(best_server, axis=(1, 2)),
        "rsrp_max": np.nanmax(best_server, axis=(1, 2)),
    },
    index=bands,
)

In [ ]:
pd.DataFrame(np.isfinite(rsrp).mean(axis=(2, 3)).T, index=tx_names, columns=bands)

In [ ]:
pd.DataFrame(radio["tilt_deg"].T, index=tx_names, columns=bands)

## 7. Synthetic MDT

In [ ]:
mdt.head()

In [ ]:
rsrp_columns = [column for column in mdt.columns if column.startswith("rsrp_")]
mdt[rsrp_columns].describe().T

In [ ]:
missing = mdt[rsrp_columns].isna().mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11.0, 4.5))
missing.plot.bar(ax=ax)
ax.set_ylabel("fraction not reported")
ax.set_title("Missingness per measurement column")
plt.show()

missing.to_frame("missing_fraction")

In [ ]:
reported = mdt[rsrp_columns].notna().sum(axis=1)

fig, ax = plt.subplots()
ax.hist(reported, bins=np.arange(reported.max() + 2) - 0.5)
ax.set_xlabel("cell-band pairs reported")
ax.set_ylabel("UEs")
ax.set_title("Measurements reported per UE")
plt.show()

In [ ]:
fig, ax = plt.subplots()
for band in bands:
    columns = [column for column in rsrp_columns if column.endswith(f"_{band}")]
    values = mdt[columns].to_numpy().ravel()
    ax.hist(values[np.isfinite(values)], bins=60, histtype="step", lw=1.4, label=band)
ax.set_xlabel("reported RSRP [dBm]")
ax.set_ylabel("measurements")
ax.set_title("Reported RSRP distribution")
ax.legend()
plt.show()

In [ ]:
pd.DataFrame(
    {
        "measurements": [
            mdt[[c for c in rsrp_columns if c.endswith(f"_{band}")]].notna().to_numpy().sum()
            for band in bands
        ],
        "reported_fraction": [
            mdt[[c for c in rsrp_columns if c.endswith(f"_{band}")]].notna().to_numpy().mean()
            for band in bands
        ],
    },
    index=bands,
)